In [21]:
import sys
from pathlib import Path

NOTEBOOK_PATH = Path().resolve()
PROJECT_ROOT = NOTEBOOK_PATH.parents[1]   # AguacAIte-Labs
sys.path.insert(0, str(PROJECT_ROOT))

import torch
import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from src.data.loaders import build_dataloaders
from src.models.densenet import build_densenet121
from src.models.resnet import build_resnet50
from src.models.efficientnet import build_efficientnet_b0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models" / "transfer"

def evaluate_model(model, dataloader):
    model.eval()

    y_true, y_pred, y_prob = [], [], []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE).unsqueeze(1)

            outputs = model(images)
            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).int()

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
            y_prob.extend(probs.cpu().numpy())

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_prob),
    }

_, val_loader = build_dataloaders(DATA_DIR)

models = {
    "DenseNet121": build_densenet121(pretrained=False, freeze_backbone=False),
    "ResNet50": build_resnet50(pretrained=False, freeze_backbone=False),
    "EfficientNet-B0": build_efficientnet_b0(pretrained=False, freeze_backbone=False),
}

weights = {
    "DenseNet121": "densenet121_finetuned.pth",
    "ResNet50": "resnet50_finetuned.pth",
    "EfficientNet-B0": "efficientnet_b0_finetuned.pth",
}

results = []

for name, model in models.items():
    weight_path = MODEL_DIR / weights[name]
    print(f"Loading {name} → {weight_path}")

    model.load_state_dict(torch.load(weight_path, map_location=DEVICE))
    model.to(DEVICE)

    metrics = evaluate_model(model, val_loader)
    metrics["model"] = name
    results.append(metrics)

# Table
df_results = pd.DataFrame(results).set_index("model")
df_results


c:\Project\AguacAIte-Labs\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Project\AguacAIte-Labs\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Loading DenseNet121 → C:\Project\AguacAIte-Labs\models\transfer\densenet121_finetuned.pth
Loading ResNet50 → C:\Project\AguacAIte-Labs\models\transfer\resnet50_finetuned.pth
Loading EfficientNet-B0 → C:\Project\AguacAIte-Labs\models\transfer\efficientnet_b0_finetuned.pth


c:\Project\AguacAIte-Labs\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


,accuracy,precision,recall,f1,roc_auc
model,,,,,
DenseNet121,0.666667,0.5,1.0,0.666667,0.750
ResNet50,0.666667,0.0,0.0,0.000000,1.000
EfficientNet-B0,0.500000,0.4,1.0,0.571429,0.875
